# The Zero Slice of Quaternionic Real Bordism 
#### Bert Guillou, Jesse Keyes, and David Mehrle

**Abstract**. This notebook contains SAGE code accompanying the paper *The Zero Slice of Quaternionic Real Bordism*. 

**WARNING**. Following the conventions of GAP for displaying the table of marks as a lower-triangular matrix, we write all vectors as *row* vectors and represent linear transformations by *right*-multiplication by matrices. Thus, everything below is the transpose of the way that the authors were taught (and teach) linear algebra with column vectors and left-multiplication. 

### Burnside Rings

The Burnside ring $A(G)$ is an important group-theoretic invariant in homotopy theory. Elements of this ring are isomorphism classes of finite $G$-sets, with addition given by disjoint union and multiplication given by cartesian product. As an abelian group $A(G)$ is generated by isomorphism classes of transitive $G$-sets $G/H$: 
$$
    A(G) \cong \bigoplus_{(H) \leq G} \mathbb{Z}\{[G/H]\},
$$
where the index of the sum runs over conjugacy classes of subgroups of $G$. 
With this choice of generators, multiplication is determined by $[G/G] = 1$ and the double coset relations: 
$$
    [G/H] \times [G/K] = \sum_{HgK \in H \backslash G / K} [G / (H \cap K^g)],
$$
where $K^g = g^{-1}Kg$. The relation $[G/G]=1$ and the double coset formula also yield a presentation of the Burnside ring as a quotient of the polynomial ring $\mathbb{Z}[x_1, \ldots, x_n]$, where $x_i = [G/H_i]$ and $H_1, \ldots, H_n$ is a choice of representatives for the conjugacy classes of subgroups of $G$. 


### Ghost Rings

Burnside rings are frequently studied via the *ghost rings*. Let $\Gamma(G)$ be the *ghost ring* 
$$
    \Gamma(G) := \prod_{(H) \leq G} \mathbb{Z}, 
$$
the product (as a ring) of one copy of $\mathbb{Z}$ for each conjugacy class of subgroups inside $G$. The ghost ring can alternatively be described as the $\mathbb{Z}$-valued class functions on subgroups of $G$: 
$$
    \Gamma(G) \cong \text{Set}(\text{Sub}(G)/G, \mathbb{Z}), 
$$
where $\text{Sub}(G)$ is the set of subgroups of $G$ and $/G$ denotes the quotient by the conjugation action, so $\text{Sub}(G)/G$ is the set of conjugacy classes of subgroups of $G$.

Note that the ghost ring is isomorphic to the Burnside ring additively, but they have different multiplications. These two rings are related by the *marks homomorphism* $\varphi \colon A(G) \to \Gamma(G)$, given by 
$$
    \varphi([X]) = (|X^H|)_{(H) \leq G },
$$
that is, the $(H)$-th coordinate of $\varphi([X])$ is the cardinality of the set of $H$-fixed points of $X$.



It is a theorem of Dress that $\varphi$ is an injective ring homomorphism. Therefore, we can study the Burnside ring by studying its image under the marks homomorphism. In fact, $\varphi \otimes \mathbb{Q} \colon A(G) \otimes \mathbb{Q} \to \Gamma(G)\otimes \mathbb{Q}$ is an isomorphism. This isomorphism furnishes this vector space $A(G) \otimes \mathbb{Q}$ with two bases: the standard basis $e_{(H)}$ coming from the basis of indicator functions for $\Gamma(G)$, and the basis of $\varphi([G/H])$ coming from the generators of $A(G)$. The *table of marks* of $G$ is the change-of-basis matrix between these two bases. If $\{H_i\}_{i=1}^n$ is a set of representatives of the conjugacy classes of subgroups of $G$, then the $i$-th row of the table of marks is $\varphi(G/H_i)$. These representatives can always be chosen such that the table of marks is a lower-triangular matrix, and we assume that this is the case in all that follows. 


### Implementation Details
It will be most useful to us to remember the ideal of relations that defines the Burnside ring, rather than the Burnside ring as a quotient. Choose representatives $H_1, \ldots, H_n$ for the conjugacy classes of subgroups of $G$, and let $x_i = [G/H_i]$. We begin with the ring $\mathbb{Z}[x_1, \ldots, x_n]$, where $n$ is the number of conjugacy classes of $G$. Let $T$ be the table of marks of $G$ as a matrix. To find the double coset relations, we compute the row vector $\vec{b} = \varphi(x_i) \varphi(x_j)$ in the Ghost ring, and then solve 
$$
    \vec{a} T = \vec{b}
$$
for the row vector $\vec{a}$, base-changing the vector $\vec{b}$ from the ghost coordinates back into the Burnside ring. That is, $\vec{a}$ is a vector of coefficients such that 
$$
    x_i \times x_j = \sum_k a_k x_k.
$$

In [1]:
# A function that computes all of the double coset relations in the Burnside ring
# 
# INPUT a group G as a GAP object
# OUTPUT an ideal I of the ring ZZ[x_0,...,x_n] where n is the number of conjugacy classes of subgroups of G
#   the generators of this ideal are the relations that define the Burnside ring, where x_i represents an orbit G/H_i
def burnsideRelations(G):

    tomG = G.TableOfMarks()
    T = matrix(ZZ,tomG.MatTom())
    n = T.dimensions()[0]

    baseRing = PolynomialRing(ZZ,'x',n) # polynomial ring with one generator per conjugacy class of subgroups
    x = baseRing.gens() # generators of this polynomial ring
    relations = [x[n-1] - 1] # add the relation x_n = 1; gap puts the trivial G/G is last in the list
    for i in range(n):
        for j in range(i,n):
            b = vector(ZZ, [T[i][k] * T[j][k] for k in range(n)]) # product \varphi(G/H_i) * \varphi(G/H_j) in the ghost ring
            a = T.solve_left(b) # solve a T = b for a, where T is table of marks
            relations.append(x[i] * x[j] - sum(coeff * x[k] for k, coeff in enumerate(a))) # add the relation x_i * x_j = \sum_k a_k x_k
    return baseRing.ideal(relations) 

# A function that computes the Burnside ring A(G)
#
# INPUT a group G as a GAP object
# OUTPUT the Burnside ring of G
def burnsideRing(G):

    tomG = G.TableOfMarks()
    T = matrix(ZZ,tomG.MatTom())
    n = T.dimensions()[0]

    baseRing = PolynomialRing(ZZ,'x',n) # polynomial ring with one generator per conjugacy class of subgroups
    return baseRing.quotient(burnsideRelations(G)) # the Burnside ring itself

### Restrictions 

The collection of Burnside rings $A(H)$ as $H$ ranges over (conjugacy classes of) subgroups of $G$ assemble into a Tambara functor called the Burnside Tambara functor. Restrictions in this Tambara functor are given by 
$$
    \text{res}^G_H([X]) = [i_H^*(X)],
$$
where $i_H^* \colon \text{Set}^G \to \text{Set}^H$ is the forgetful functor. 


The Ghost rings $\Gamma(H)$ also assemble into a Tambara functor $\underline{\Gamma}$ and the marks homomorphism $\varphi \colon \underline{A} \to \underline{\Gamma}$ is a homomorphism of Tambara functors [[CCM+](https://academic.oup.com/imrn/article-abstract/2026/2/rnaf388/8436054?redirectedFrom=PDF), Theorems 3.13 and 3.14]. The restriction in $\underline{\Gamma}$ is simply a projection onto those coordinates indexed by conjugacy classes of subgroups of $H$ [[CCM+](https://academic.oup.com/imrn/article-abstract/2026/2/rnaf388/8436054?redirectedFrom=PDF), Definition 2.25].

### Implementation Details

In the ghost Tambara functor, the restriction is described by a projection matrix that forgets some basis vectors. To find the matrix for restriction in the Burnside Tambara functor, we base-change using the table of marks of both $H$ and $G$. 

In [2]:
# A function to compute the restriction from G to H in the ghost ring
#
# INPUT 
# H,G are GAP groups such that H is a subgroup of G
#
# OUTPUT
# A matrix M such that right-multiplication by M implements the restriction from G to H
# in the ghost ring. 
#
# Note that the order of the inputs (bigger group first, subgroup second) is opposite the transfer and norm below  
def restriction_marks(G,H):
    # if H is not a subgroup of G, then throw an error 
    if not gap.IsSubgroup(G,H): 
        raise ValueError("You fool! H must be a subgroup of G") # maybe the error could be nicer

    tomG = G.TableOfMarks() # create the table of marks of G
    matG = matrix(ZZ, tomG.MatTom()) # table of marks as a matrix 
    nG = matG.dimensions()[0] # dimensions of table of marks
    repsG = [tomG.RepresentativeTom(n+1) for n in range(nG)] # reps of conjugacy classes

    tomH = H.TableOfMarks() # create the table of marks of H
    matH = matrix(ZZ, tomH.MatTom()) # table of marks as a matrix
    nH = matH.dimensions()[0]
    repsH = [tomH.RepresentativeTom(n+1) for n in range(nH)] # reps of conjugacy classes

    return matrix(ZZ,nG,nH,lambda i,j : G.IsConjugate(repsG[i],repsH[j]).sage())
        # G.IsConjugate(K1,K2) returns a boolean GAP object; .sage() converts it to normal True/False in python
        # then python interprets True/False as 0 or 1 in a matrix.


# A function to compute there striction from G to H in the Burnside ring
#
# INPUT 
# H,G are GAP groups such that H is a subgroup of G
#
# OUTPUT
# A matrix M that represents the restriction from A(G) to A(H).
# RIGHT-multiplication by this matrix M of a row vector in A(G) represents restriction to A(H).
# These Burnside rings are given the additive bases of orbits (as row vectors), 
# presented in the same order they appear in the rows of the GAP table of marks.
#
# Note that the order of the inputs (bigger group first, subgroup second) is opposite the transfer and norm below  
def restriction(G,H):
    # if H is not a subgroup of G, then throw an error 
    if not gap.IsSubgroup(G,H): 
        raise ValueError("You fool! H must be a subgroup of G") # maybe the error could be nicer

    tomG = G.TableOfMarks() # create the table of marks of G
    matG = matrix(ZZ, tomG.MatTom()) # table of marks as a matrix 
    
    tomH = H.TableOfMarks() # create the table of marks of H
    matH = matrix(ZZ, tomH.MatTom()) # table of marks as a matrix

    return matG*restriction_marks(G,H)*matH^(-1)

### Transfers

Transfers $\text{tr}_H^G \colon A(H) \to A(G)$ in the Burnside Tambara functor are given by 
$$ 
    \text{tr}_H^G([X]) = \left[X \times_H G\right].
$$
In particular, the transfer from $H$ to $G$ of $H/K$ is $G/K$. 

Transfers also have a nice formula in the ghost ring [[CCM+](https://academic.oup.com/imrn/article-abstract/2026/2/rnaf388/8436054?redirectedFrom=PDF), Definition 2.25]. For an arbitrary class function $f \colon \text{Sub(G)}/G \to \mathbb{Z}$, 
$$
    \text{tr}_H^G(f)([K]) = \sum_{\substack{gH \in G/H \\ K^g \subseteq H}} f([K^g])
$$


### Implementation Details
In this case, it is easier to implement the transfers in the Burnside Tambara functor; it's simply a binary matrix. While we could simply obtain the transfer matrices in the ghost Tambara functor by base change, we implement it independently as a way to check the correctness of these two functions against each other.

In [3]:
# A function to compute the transfer from H to G in the Burnside ring
#
# INPUT
# H,G are GAP groups such that H is a subgroup of G
#
# OUTPUT
# A matrix M that represents the transfer from A(H) to A(G).
# RIGHT-multiplication by this matrix M of a row vector in A(H) represents transfer to A(G).
# These Burnside rings are given the additive bases of orbits (as row vectors), 
# presented in the same order they appear in the rows of the GAP table of marks.
#
def transfer(H,G):
    # if H is not a subgroup of G, then throw an error 
    if not gap.IsSubgroup(G,H): 
        raise ValueError("You fool! H must be a subgroup of G") # maybe the error could be nicer
    
    tomG = G.TableOfMarks() # create the table of marks of G
    matG = matrix(ZZ, tomG.MatTom()) # table of marks as a matrix 
    nG = matG.dimensions()[0] # dimensions of table of marks
    repsG = [tomG.RepresentativeTom(n+1) for n in range(nG)] # reps of conjugacy classes

    tomH = H.TableOfMarks() # create the table of marks of H
    matH = matrix(ZZ, tomH.MatTom()) # table of marks as a matrix
    nH = matH.dimensions()[0]
    repsH = [tomH.RepresentativeTom(n+1) for n in range(nH)] # reps of conjugacy classes

    return matrix(ZZ,nH,nG,lambda i,j: G.IsConjugate(repsH[i],repsG[j]).sage()) 
        # create a matrix where the i,j entry is 1 if G \times_H H/K_i = G/L_j
        # G.IsConjugate(K1,K2) returns a boolean GAP object; .sage() converts it to normal True/False
        # then python interprets True/False as 0 or 1 in a matrix.

# A function that implements the transfer on the ghost ring (as above)
#   This is strictly speaking not necessary, because you could just get 
#   this matrix by basis-changing the matrix of transfer(H,G) using the tables of marks.
#   In fact, it's almost certainly more efficient to get it that way
#
# INPUT 
# H,G are GAP groups such that H is a subgroup of G
#
# OUTPUT
# A matrix M such that right-multiplication by M implements the transfer from H to G
# in the ghost ring. 
#
def transfer_marks(H,G):
    # if H is not a subgroup of G, then throw an error 
    if not gap.IsSubgroup(G,H): 
        raise ValueError("You fool! H must be a subgroup of G") # maybe the error could be nicer

    tomG = G.TableOfMarks() # create the table of marks of G
    matG = matrix(ZZ, tomG.MatTom()) # table of marks as a matrix 
    nG = matG.dimensions()[0] # dimensions of table of marks
    repsG = [tomG.RepresentativeTom(n+1) for n in range(nG)] # reps of conjugacy classes

    tomH = H.TableOfMarks() # create the table of marks of H
    matH = matrix(ZZ, tomH.MatTom()) # table of marks as a matrix
    nH = matH.dimensions()[0]
    repsH = [tomH.RepresentativeTom(n+1) for n in range(nH)] # reps of conjugacy classes

    cosetReps = [C.Representative() for C in G.LeftCosets(H)] # coset representatives for G/H

    tr = matrix(ZZ,nH,nG) # make a matrix of all zeros of the appropriate dimensions

    for i in range(nH):
        L = repsH[i]
        for j in range(nG): 
            K = repsG[j]
            tr[i,j] = sum(1 for g in cosetReps if H.IsConjugate(L,K.ConjugateGroup(g)))
    return tr

### Norms
The norm $\text{nm}_H^G \colon A(H) \to A(G)$ in the Burnside Tambara functor is given by 
$$
    \text{nm}_H^G([X]) = \left[\text{Set}^H(G,X)\right],
$$
where the right-hand side is the $H$-equivariant functions from $G$ to $X$. This becomes a $G$-set via $$(g \cdot f)(x) = f(xg).$$ 

We can also encode these norms in the table of marks [[CCM+](https://academic.oup.com/imrn/article-abstract/2026/2/rnaf388/8436054?redirectedFrom=PDF), Definition 2.25]. For an arbitrary class function $f \colon \text{Sub}(G)/G \to \mathbb{Z}$, 
$$
    \text{nm}_H^G(f)([K]) = \prod_{KgH \in K \backslash G / H} f([K^g \cap H]).
$$

### Implementation Details
The norm is not a linear function, and therefore cannot be represented by a matrix. So we take as input a vector $f$ representing a linear combination of orbits in the Burnside ring or a row vector in the ghost ring. While we could in principle ask GAP to decompose $\text{Set}^H(G,X)$ into orbits, it's significantly faster to use the formula in the ghost ring and use the table of marks as a change-of-basis matrix. Because base-changing back and forth between the bases adds time complexity, we implement the algorithm twice to allow for inputs either in $\Gamma(H)$ or $A(H)$. 

Finally, because the norm is not linear, we provide a function to print a formula for the norm of a sum in the Burnside Tambara functor. As a warning, the displayed output of `print_symbolic_norm` does not distinguish between abstractly isomorphic subgroups. The formula will be correct, but some interpretation will be required. For example, computing the symbolic norm from $e$ to $C_2 \times C_2$ will display three formulas labelled with "coefficients of $C_2 \times C_2/C_2$," corresponding to the three non-conjugate copies of $C_2$ inside $C_2 \times C_2$.

The authors thank Ben Spitz, David Chan, and Chase Vogeli for their help with the implementation of the functions below.

In [4]:
# A function to compute the norm from H to G in the Burnside ring
#
# INPUT
# H, G are GAP groups
# f is a vector defining a linear combination of H-orbits
#   The order of orbits is the same as the rows of the table of marks of H, as given by GAP
#  
#   For the quaternion group Q_8, the GAP table of marks presents conjugacy classes of subgroups in the order 
#   e, C_2, I, J, K, Q_8, where I = <i>, J = <j>, and K = <k>. 
#
#   For example, the vector f = (0,1,-1,0,2,0) represents the element Q_8/C_2 - Q_8/I + 2 Q_8/K
#
# OUTPUT 
# a vector nm encoding a the norm from H to G of f as a linear combination of G-orbits, 
# with order of orbits as above
#
#
def norm(H,G,f):
    # if H is not a subgroup of G, then throw an error 
    if not gap.IsSubgroup(G,H): 
        raise ValueError("You fool! H must be a subgroup of G") # maybe the error could be nicer

    tomG = G.TableOfMarks() # create the table of marks of G
    matG = matrix(ZZ, tomG.MatTom()) # table of marks as a matrix 
    nG = matG.dimensions()[0] # dimension of table of marks
    repsG = [tomG.RepresentativeTom(n+1) for n in range(nG)] # reps of conjugacy classes

    tomH = H.TableOfMarks() # create the table of marks of H
    matH = matrix(ZZ, tomH.MatTom()) # table of marks as a matrix
    nH = matH.dimensions()[0]
    repsH = [tomH.RepresentativeTom(n+1) for n in range(nH)] # reps of conjugacy classes

    marksf = f * matH # convert the vector f of coefficients into an element of the ghost ring
    nm = vector(ZZ, [1]*nG) # empty product is 1; setup for looping
    for i in range(nG):
        K = repsG[i] # we're finding the component indexed by K
        for(g,_) in G.DoubleCosetRepsAndSizes(K,H): # only need the reps of double cosets
            HcapKg = gap.Intersection(H,K.ConjugateGroup(g)) # form the intersection of H and the conjugate of K by g
            index = next(j for j,r in enumerate(repsH) if H.IsConjugate(r, HcapKg)) # find the index of the subgroup of H that represents H \cap K^g
            nm[i] *= marksf[index] # multiply the [K] component of the norm by marksf([H \cap K^g])
    return matG.solve_left(nm) # convert back into a list of coefficients



# A function to compute the norm from H to G in the ghost ring
#
# INPUT
# H, G are GAP groups
# f is a row vector in the Ghost ring of H
#
# OUTPUT 
# the norm from H to G of f in the Ghost ring of G
#
def norm_marks(H,G,f):
    # if H is not a subgroup of G, then throw an error 
    if not gap.IsSubgroup(G,H): 
        raise ValueError("You fool! H must be a subgroup of G") # maybe the error could be nicer

    tomG = G.TableOfMarks() # create the table of marks of G
    matG = matrix(ZZ, tomG.MatTom()) # table of marks as a matrix 
    nG = matG.dimensions()[0] # dimension of table of marks
    repsG = [tomG.RepresentativeTom(n+1) for n in range(nG)] # reps of conjugacy classes

    tomH = H.TableOfMarks() # create the table of marks of H
    matH = matrix(ZZ, tomH.MatTom()) # table of marks as a matrix
    nH = matH.dimensions()[0]
    repsH = [tomH.RepresentativeTom(n+1) for n in range(nH)] # reps of conjugacy classes

    nm = vector(ZZ, [1]*nG) # empty product is 1; setup for looping
    for i in range(nG):
        K = repsG[i] # we're finding the component indexed by K
        for(g,_) in G.DoubleCosetRepsAndSizes(K,H): # only need the reps of double cosets
            HcapKg = gap.Intersection(H,K.ConjugateGroup(g)) # form the intersection of H and the conjugate of K by g
            index = next(j for j,r in enumerate(repsH) if H.IsConjugate(r, HcapKg)) # find the index of the subgroup of H that represents H \cap K^g
            nm[i] *= f[index] # multiply the [K] component of the norm by marksf([H \cap K^g])
    return nm

# A function to print a formula for the norm from H to G in the Burnside Tambara functor
#
# INPUT
# H, G are GAP groups
# f is a row vector in the Ghost ring of H
#
# OUTPUT 
# none 
#
# EFFECTS 
# prints a formula for the norm from H to G in the Burnside Tambara functor
#
def print_symbolic_norm(H,G):
    # if H is not a subgroup of G, then throw an error 
    if not gap.IsSubgroup(G,H): 
        raise ValueError("You fool! H must be a subgroup of G") # maybe the error could be nicer

    tomG = G.TableOfMarks() # create the table of marks of G
    matG = matrix(ZZ, tomG.MatTom()) # table of marks as a matrix 
    nG = matG.dimensions()[0] # dimensions of the table of marks
    repsG = [tomG.RepresentativeTom(n+1) for n in range(nG)] # reps of conjugacy classes

    tomH = H.TableOfMarks() # create the table of marks of H
    matH = matrix(ZZ, tomH.MatTom()) # table of marks as a matrix
    nH = matH.dimensions()[0] # dimension of the table of marks
    repsH = [tomH.RepresentativeTom(n+1) for n in range(nH)] # reps of conjugacy classes
    
    R = PolynomialRing(QQ, nH, "a") # polynomial ring with nH many generators
    a = vector(R, R.gens()) # vector over symbolic ring R

    marksa = a * matH # convert the vector f of coefficients into an element of the ghost ring
    nm = vector(R, [1]*nG) # empty product is 1; setup for looping
    for i in range(nG):
        K = repsG[i] # we're finding the component indexed by K
        for(g,_) in G.DoubleCosetRepsAndSizes(K,H): # only need the reps of double cosets
            HcapKg = gap.Intersection(H,K.ConjugateGroup(g)) # form the intersection of H and the conjugate of K by g
            index = next(j for j,r in enumerate(repsH) if H.IsConjugate(r, HcapKg)) # find the index of the subgroup of H that represents H \cap K^g
            nm[i] *= marksa[index] # multiply the [K] component of the norm by marksf([H \cap K^g])
    nm = matG.solve_left(nm) # convert back into a list of coefficients
    
    print("============ Norm from", gap.StructureDescription(H), "to", gap.StructureDescription(G), "============\n")    
    print("Norm of\n")

    for i in range(nH):
        if i == 0:
            print('\t',end='')
        print(a[i]," "+str(gap.StructureDescription(H)),"/",str(gap.StructureDescription(repsH[i])),sep='',end='')
        if i < nH-1:
            print(" + ",end='')
    
    print("\n\nhas coefficients:\n")
        
    for i in range(nG):
        print("\tcoefficient of " + str(gap.StructureDescription(G)) + "/" + str(gap.StructureDescription(repsG[i])) + ":",end='')
        print("\t",factor(nm[i])) # dunno if factor is better than not
        # uncomment this line if you want latex output
        # long term we should probably add a ton of optional argumnets to this
        # show(nm[i]) 

## Examples 
If you are looking for a particular group, [https://people.maths.bris.ac.uk/~matyd/GroupNames/](https://people.maths.bris.ac.uk/~matyd/GroupNames/) is a great reference for the locations of these groups in the GAP SmallGroups library.

We start with the trivial example $A(e) \cong \mathbb{Z}$. Our algorithm computes this ring as a quotient of $\mathbb{Z}[x_0]$ by the double coset relations, where $x_0$ is the class of $[e/e]$. 

In [5]:
e = gap("TrivialGroup()")
burnsideRelations(e)

Ideal (x - 1, x^2 - x) of Multivariate Polynomial Ring in x over Integer Ring

This next example computes the Burnside ring of a group of order 2 as the quotient of $\mathbb{Z}[x_0,x_1]$ by the double coset relations. Here, $x_0$ represents the class of $C_2/e$ and $x_1$ represents the class of $C_2/C_2$. Again, we compute a Gröbner basis to eliminate redundancy among the ideal of relations.

In [6]:
C2 = gap("CyclicGroup(2)")
burnsideRelations(C2).groebner_basis() 

[x0^2 - 2*x0, x1 - 1]

We can finish the computation of the $C_2$-Burnside Tambara functor by describing the restriction, transfer, and norm.

In [7]:
e = C2.AllSubgroups()[1] # as constructed above, e is not literally a subset of C2, so this is a workaround
print("======== Restriction from C2 to e =========")
print(restriction(C2,e),"\n")
print("========== Transfer from C2 to e ==========")
print(transfer(e,C2),"\n")
print_symbolic_norm(e,C2)


======== Restriction from C2 to e =========
[2]
[1] 

========== Transfer from C2 to e ==========
[1 0] 

============ Norm from 1 to C2 ============

Norm of

	a 1/1

has coefficients:

	coefficient of C2/1:	 (1/2) * (a - 1) * a
	coefficient of C2/C2:	 a


The next example was discussed in the implementation details of the function `print_symbolic_norm`. There are three copies of $C_2$ inside $C_2 \times C_2$, none of which are conjugate to each other. Because the structure description provided by GAP doesn't distinguish between isomorphic copies of a group, the output of `print_symbolic_norm(e,C2timesC2)` contains three lines described as "coefficient of C2 x C2/C2". Each line corresponds to one of the copies of $C_2$ inside $C_2 \times C_2$; these are three separate terms, rather than redundancy in the output or coefficients that should be added together. 

Concretely, this example shows that the norm from $e$ to $C_2$ of an integer $a \in A(e) \cong \mathbb{Z}$ is given by 
$$
    \frac{a(a+2)(a-1)^2}{4} [C_2 \times C_2/e] + \frac{a(a-1)}{2} [C_2 \times C_2/L] + \frac{a(a-1)}{2} [C_2 \times C_2/D] + \frac{a(a-1)}{2} [C_2 \times C_2/R] + a [C_2 \times C_2 / C_2 \times C_2],
$$
where $L, D, R$ are the three order 2 subgroups of $C_2 \times C_2$. 

In [8]:
# the example discussed above
C2timesC2 = gap("SmallGroup(4,2)")
e = C2timesC2.AllSubgroups()[1]
print_symbolic_norm(e,C2timesC2)


============ Norm from 1 to C2 x C2 ============

Norm of

	a 1/1

has coefficients:

	coefficient of C2 x C2/1:	 (1/4) * a * (a + 2) * (a - 1)^2
	coefficient of C2 x C2/C2:	 (1/2) * (a - 1) * a
	coefficient of C2 x C2/C2:	 (1/2) * (a - 1) * a
	coefficient of C2 x C2/C2:	 (1/2) * (a - 1) * a
	coefficient of C2 x C2/C2 x C2:	 a


While most mathematicians working in this area would recognize the Burnside rings presented above, we can also compute some unfamiliar Burnside rings. This example takes about 45 seconds on an M1 Mac Mini. We forego the Gröbner basis computation because it takes a while longer; the implementation of Buchberger's algorithm in SAGE is quite efficient, but it is still exponential on the size of the input.  

In [9]:
S7 = gap("SymmetricGroup(7)")
burnsideRelations(S7)

Ideal (x95 - 1, x0^2 - 5040*x0, x0*x1 - 2520*x0, x0*x2 - 2520*x0, x0*x3 - 2520*x0, x0*x4 - 1680*x0, x0*x5 - 1680*x0, x0*x6 - 1260*x0, x0*x7 - 1260*x0, x0*x8 - 1260*x0, x0*x9 - 1260*x0, x0*x10 - 1260*x0, x0*x11 - 1260*x0, x0*x12 - 1260*x0, x0*x13 - 1008*x0, x0*x14 - 840*x0, x0*x15 - 840*x0, x0*x16 - 840*x0, x0*x17 - 840*x0, x0*x18 - 840*x0, x0*x19 - 840*x0, x0*x20 - 840*x0, x0*x21 - 840*x0, x0*x22 - 720*x0, x0*x23 - 630*x0, x0*x24 - 630*x0, x0*x25 - 630*x0, x0*x26 - 630*x0, x0*x27 - 630*x0, x0*x28 - 630*x0, x0*x29 - 630*x0, x0*x30 - 560*x0, x0*x31 - 504*x0, x0*x32 - 504*x0, x0*x33 - 504*x0, x0*x34 - 420*x0, x0*x35 - 420*x0, x0*x36 - 420*x0, x0*x37 - 420*x0, x0*x38 - 420*x0, x0*x39 - 420*x0, x0*x40 - 420*x0, x0*x41 - 420*x0, x0*x42 - 420*x0, x0*x43 - 420*x0, x0*x44 - 420*x0, x0*x45 - 420*x0, x0*x46 - 420*x0, x0*x47 - 360*x0, x0*x48 - 315*x0, x0*x49 - 280*x0, x0*x50 - 280*x0, x0*x51 - 280*x0, x0*x52 - 252*x0, x0*x53 - 252*x0, x0*x54 - 252*x0, x0*x55 - 240*x0, x0*x56 - 210*x0, x0*x57 - 210

## Lemmas, Propositions, and Examples from the paper

We verify some computations in section 2 of our paper using the code above.

### **Lemma 2.7**. 
In the $C_4$-Burnside Tambara functor, the norm of $t - 2$ is given by 
$$
    \text{nm}_{C_2}^{C_4}(t-2) = -[C_4/e] + 3[C_4/C_2] - 2,
$$
where $t = [C_2/e]$. 

In [10]:
C4 = gap("CyclicGroup(4)")
C2 = C4.AllSubgroups()[2]

t = vector(ZZ,[1,0]) 
tminus2 = t - vector(ZZ,[0,2])
norm_tminus2 = norm(C2,C4,tminus2) ## this gives the result of the lemma

print("Norm of t - 2 is",norm_tminus2)

Norm of t - 2 is (-1, 3, -2)


In the proof of Lemma 2.7, we also compute some other norms as well.

In [11]:
norm_2 = norm(C2,C4,vector(ZZ,[0,2])) # norm of 2
norm_minus1 = norm(C2,C4,vector(ZZ,[0,-1])) # norm of -1
norm_t = norm(C2,C4,t) # norm of t
transfer_minus2t = vector(ZZ,[-2,0])*transfer(C2,C4)

print("Norm of 2 is",norm_2)
print("Norm of -1 is",norm_minus1)
print("Norm of t is",norm_t)
print("Transfer of -2t is",transfer_minus2t)


Norm of 2 is (0, 1, 2)
Norm of -1 is (0, 1, -1)
Norm of t is (1, 0, 0)
Transfer of -2t is (-2, 0, 0)


### **Proposition 2.8**. 
Let $C_4$ be any order 4 subgroup of $Q_8$. Then $(\underline{A}/\underline{\mathfrak{a}})(Q_8/C_4)$ is isomorphic to $\mathbb{Z}$ as a commutative ring.

In the proof of this proposition, we use the transfer of $t-2$.


In [12]:
transfer_tminus2 = tminus2*transfer(C2,C4)

print("Transfer of t - 2 is", transfer_tminus2)

Transfer of t - 2 is (1, -2, 0)


In the same proof, we also rewrite $[C_4/C_2] - 2$ as a linear combination of $\text{tr}_{C_2}^{C_4}(t-2)$ and $\text{nm}_{C_2}^{C_4}(t-2)$. Here, $[C_4/C_2] - 2$ corresponds to the row vector `(0,1,-2)`, $\text{tr}_{C_2}^{C_4}(t-2)$ is `(1,-2,0)`, and $\text{nm}_{C_2}^{C_4}(t-2)$ is `(-1,3,-2)`. Similarly, we also rewrite $[C_4/e] - 4$, corresponding to the row vector `(1,0,-4)`, as a linear combination of that same transfer and norm.

In [13]:
M = matrix(ZZ,[transfer_tminus2,norm_tminus2])
print("[C4/C2] - 2 as a linear combination of the rows of M:",M.solve_left(vector([0,1,-2])))
print(" [C4/e] - 4 as a linear combination of the rows of M:",M.solve_left(vector([1,0,-4])))

[C4/C2] - 2 as a linear combination of the rows of M: (1, 1)
 [C4/e] - 4 as a linear combination of the rows of M: (3, 2)


There is another way to reach the conclusion of Proposition 2.8 that is more amenable to code. By Remark 2.9, the ideal $\underline{\mathfrak{a}}(Q_8/C_4)$ is generated by $\text{tr}_{C_2}^{C_4}(t-2)$ and $\text{nm}_{C_2}^{C_4}(t-2)$. Then 
$$
    A(C_4)/\underline{\mathfrak{a}}(Q_8/C_4) \cong \mathbb{Z}[x_0,x_1,x_2]/(\mathfrak{a}(Q_8/C_4) + \mathfrak{b}),
$$
where $\mathfrak{b}$ is the ideal of relations in the Burnside ring and $x_0 = [C_4/e]$, $x_1 = [C_4/C_2]$, and $x_2 = [C_4/C_4]$. 

In [14]:
baseRing = PolynomialRing(ZZ,'x',3)
x = baseRing.gens()
b = burnsideRelations(C4)
aQ8modC4gens = [transfer_tminus2, norm_tminus2]
a = baseRing.ideal([sum(f[i]*x[i] for i in range(3)) for f in aQ8modC4gens])
(a + b).groebner_basis()

[x0 - 4, x1 - 2, x2 - 1]

The above shows that 
$$
    A(C_4)/\underline{\mathfrak{a}}(Q_8/C_4) \cong \mathbb{Z}[x_0,x_1,x_2] / (x_0 - 4, x_1 - 2, x_2 - 1) \cong \mathbb{Z},
$$
as desired.

### **Lemma 2.12**. 
In the $Q_8$-Burnside Tambara functor, the norm of $t-2$ is given by 
$$
    \text{nm}_{C_2}^{Q_8}(t - 2) = 
		-2 [{Q_8}/{e}] + 3 [{Q_8}/{I}] + 3 [{Q_8}/{J}] + 3 [{Q_8}/{K}] -2 
$$

In [15]:
Q = gap("QuaternionGroup(8)")
repsQ = [Q.TableOfMarks().RepresentativeTom(n+1) for n in range(6)]
e,C2,I,J,K,_ = repsQ # give names to the subgroups. Last one is already Q
assert Q == repsQ[5] # double check

normCQ_tminus2 = norm(C2,Q,tminus2) ## result of the lemma

print("Norm from C_2 to Q_8 of t - 2 is ", normCQ_tminus2)

Norm from C_2 to Q_8 of t - 2 is  (-2, 0, 3, 3, 3, -2)


The proof also contains the following intermediate computations.

In [16]:
normIQ_Imode = norm(I,Q,vector([1,0,0]))
normIQ_ImodC2 = norm(I,Q,vector([0,1,0]))

print("Norm from I to Q_8 of [I/e] is ",normIQ_Imode)
print("Norm from I to Q_8 of [I/C2] is ",normIQ_ImodC2)

Norm from I to Q_8 of [I/e] is  (2, 0, 0, 0, 0, 0)
Norm from I to Q_8 of [I/C2] is  (0, 0, 0, 1, 1, 0)


Prior to the proof of Lemma 2.12, we learned that the ideal $\underline{\mathfrak{a}}(Q_8/Q_8)$ is generated by the elements
$$
\begin{align*}
    \text{nm}_{C_2}^{Q_8}(t-2) &= -2 [{Q_8}/{e}] + 3 [{Q_8}/{I}] + 3 [{Q_8}/{J}] + 3 [{Q_8}/{K}] -2\\
    \text{tr}_{C_2}^{Q_8}(t-2) &= [{Q_8}/{e}] - 2 [{Q_8}/{C_2}]\\
    \text{tr}_I^{Q_8} \text{nm}_{C_2}^I(t-2) &= -[{Q_8}/{e}] + 3 [{Q_8}/{C_2}] -2 [{Q_8}/{I}]\\
    \text{tr}_J^{Q_8} \text{nm}_{C_2}^J(t-2) &= -[{Q_8}/{e}] + 3 [{Q_8}/{C_2}] -2 [{Q_8}/{J}]\\
    \text{tr}_K^{Q_8} \text{nm}_{C_2}^K(t-2) &= -[{Q_8}/{e}] + 3 [{Q_8}/{C_2}] -2 [{Q_8}/{K}]
\end{align*}
$$
We can express these generators in code.

In [17]:
transferCQ_tminus2 = tminus2*transfer(C2,Q)
transferIQ_normCI_tminus2 = norm(C2,I,tminus2)*transfer(I,Q)
transferJQ_normCJ_tminus2 = norm(C2,J,tminus2)*transfer(J,Q)
transferKQ_normCK_tminus2 = norm(C2,K,tminus2)*transfer(K,Q)

print("Norm from C2 to Q8 of t - 2 is",normCQ_tminus2) # Lemma 2.12
print("Transfer from C2 to Q8 of t-2 is",transferCQ_tminus2)
print("Transfer from I to Q of Norm from C2 to I of t -2 is",transferIQ_normCI_tminus2)
print("Transfer from J to Q of Norm from C2 to J of t -2 is",transferJQ_normCJ_tminus2)
print("Transfer from K to Q of Norm from C2 to K of t -2 is",transferKQ_normCK_tminus2)

Norm from C2 to Q8 of t - 2 is (-2, 0, 3, 3, 3, -2)
Transfer from C2 to Q8 of t-2 is (1, -2, 0, 0, 0, 0)
Transfer from I to Q of Norm from C2 to I of t -2 is (-1, 3, -2, 0, 0, 0)
Transfer from J to Q of Norm from C2 to J of t -2 is (-1, 3, 0, -2, 0, 0)
Transfer from K to Q of Norm from C2 to K of t -2 is (-1, 3, 0, 0, -2, 0)


### **Proposition 2.14**. 
The quotient of $\underline{A}(Q_8/Q_8)$ by $\underline{\mathfrak{a}}(Q_8/Q_8)$ is 
$$
    \underline{A}/\underline{\mathfrak{a}}(Q_8/Q_8) \cong \mathbb{Z}[\bar u_I, \bar u_K] / (\bar u_I^2, \bar u_K^2, \bar u_I \bar u_K, 2 \bar u_I, 2 \bar u_K),
$$
where $\bar u_I$ is the image of $[{Q_8}/{I}] - 2$ and $\bar u_K$ is the image of $[{Q_8}/{K}] - 2$

In the proof of Proposition 2.14, we define a ring homomorphism $q \colon A(Q_8) \to \mathbb{Z}[\bar u_I, \bar u_K, u_Iu_K, 2\bar u_I, 2 \bar u_K]$ by 
$$
\begin{align*}
    q([Q_8/e]) &= 8\\
    q([Q_8/C_4]) &= 4\\
    q([Q_8/I]) &= \bar u_I + 2\\
    q([Q_8/J]) &= \bar u_I + \bar u_J + 2\\
    q([Q_8/K]) &= \bar u_K + 2.
\end{align*}
$$
We also compute its kernel. Below, we implement this homomorphism (as a group homomorphism) and compute its kernel. 

In [18]:
# Compute the kernel of the map A(Q_8) --> R using group homomorphisms
AQ8group = AdditiveAbelianGroup([0 for H in repsQ]) # one copy of Z for each orbit type of Q8
R = AdditiveAbelianGroup([0,2,2]) 
    # Z + Z/2 + Z/2; additive abelian group of R = Z[uI,uK]/(uI^2,uK^2,uIuK,2uI,2uK)
one,uI,uK = R.gens()
q = AQ8group.hom([8*one,4*one,uI+uK+2*one,uI+2*one,uK+2*one,one])
r = [vector(g.lift()) for g in q.kernel().gens()]; r
#q.kernel is again an additive abelian group, in this case isomorphic to Z^5
# the generators of the kernel are therefore represented by standard basis [1,0,0,0,0], etc.
# we must take lifts of generators to understand them as elements of the Burnside ring

[(1, 0, 0, 0, 0, -8),
 (0, 1, 0, 0, 0, -4),
 (0, 0, 1, 1, 1, -6),
 (0, 0, 0, 2, 0, -4),
 (0, 0, 0, 0, 2, -4)]

In the paper, we use a slightly different set of generators for the kernel, obtained by row operations on the matrix $R$ whose rows are generators of the kernel. The particular row operations below are reversible over $\mathbb{Z}$, and therefore do not change the rowspace of $R$.

In [19]:
R = matrix(r)
R.add_multiple_of_row(0,1,-2) # add -2 * row 1 to row 0 
R.add_multiple_of_row(1,3,-1) # add -1 * row 3 to row 1
R

[ 1 -2  0  0  0  0]
[ 0  1  0 -2  0  0]
[ 0  0  1  1  1 -6]
[ 0  0  0  2  0 -4]
[ 0  0  0  0  2 -4]

We can test if these generators are in the ideal $\underline{\mathfrak{a}}(Q_8/Q_8)$. In fact, they lie in the $\mathbb{Z}$-span of the generators, as the following linear algebra shows. 

In [20]:
M = matrix([normCQ_tminus2,transferCQ_tminus2,transferIQ_normCI_tminus2,transferJQ_normCJ_tminus2,transferKQ_normCK_tminus2])
M.solve_left(R) # find a matrix X such that R = X*M
# the rows of X express rows of R as linear combinations of rows of M
# in other words, express generators of the kernel of q as linear combinations of generators of the ideal a(Q_8/Q_8)

[ 0  1  0  0  0]
[ 0  1  0  1  0]
[ 3 18  4  4  4]
[ 2 12  3  2  3]
[ 2 12  3  3  2]

We can also arrive at the conclusion of Proposition 2.14 in another way. The quotient $\underline{A}(Q_8/Q_8) / \underline{\mathfrak{a}}(Q_8/Q_8)$ is isomorphic to the quotient $\mathbb{Z}[x_0,x_1,\ldots,x_5] / \mathfrak{b} + \mathfrak{\underline{a}}(Q_8/Q_8)$, where $\mathfrak{b}$ is the defining ideal of the Burnside ring and the variables $x_i$ correspond to the orbits of $Q_8$ in the order $Q_8/e, Q_8/C_2, Q_8/I, Q_8/J, Q_8/K, Q_8/Q_8$. 

In [21]:
baseRing = PolynomialRing(ZZ,'x',6)
x = baseRing.gens()
b = burnsideRelations(Q)
aQmodQgens = [normCQ_tminus2,transferCQ_tminus2,transferIQ_normCI_tminus2,transferJQ_normCJ_tminus2,transferKQ_normCK_tminus2]
a = baseRing.ideal([sum(f[i]*x[i] for i in range(6)) for f in aQmodQgens])
(a + b).groebner_basis() #compute a Groebner basis of the sum of ideals a and b

[x3^2 - 4,
 x3*x4 - 4,
 x4^2 - 4,
 x0 - 8,
 x1 - 4,
 x2 + x3 - 3*x4 + 2,
 2*x3 - 4,
 2*x4 - 4,
 x5 - 1]

The output of the previous code block is in terms of the variables $x_0, \ldots, x_5$, but in Proposition 2.14 we make the change of variables $u_2 = x_2-2$, $u_3 = x_3 - 2$, and $u_4 = x_4 - 2$ corresponding to $u_2 = [Q_8/I] - 2$, $u_3 = [Q_8/J] - 2$, and $u_4 = [Q_8/J]-2$. In the paper, $u_2$ is $u_I$, $u_3$ is $u_J$, and $u_4$ is $u_K$. 

In [22]:
u = [x[0],x[1],x[2]-2,x[3]-2,x[4]-2,x[5]] # the change of variables x \mapsto u

ideal2 = baseRing.ideal([u[0]-8,u[1]-4,u[5]-1,u[2]^2,u[4]^2,u[2]*u[4],2*u[2],2*u[4],u[2]+u[4]-u[3]])

ideal2 == a+b # check that the ideal is the same after change of variables

True

With this computation, we see that
$$
   (\underline{A}/\underline{\mathfrak{a}})(Q_8/Q_8) \cong \mathbb{Z}[u_0,u_1,u_2,u_3,u_4,u_5] / (u_0 - 8, u_1 - 4, u_5 - 1, u_2^2, u_4^2, u_2u_4, 2u_2, 2u_4, u_2 + u_4 - u_3),
$$
and after getting rid of extra variables, we arrive at 
$$
    \mathbb{Z}[\bar u_2, \bar u_4]/(\bar u_2^2,\bar u_4^2,\bar u_2 \bar u_4,2\bar u_2,2 \bar u_4),
$$
as in the statement of the proposition.